# Computação Paralela — Aula 2
## Modelos de Programação Paralela em Python

**Fork-Join • SPMD • DAG • Threads • Processos • `Executor`/`Future` • GIL**  
Curso Superior de Tecnologia em Inteligência Artificial — FATESG-GO

### Situação profissional
Uma integradora de IA recebe lotes crescentes de arquivos de biometria/inspeção. Para cada item, o pipeline precisa **ler bytes**, executar um **cálculo de assinatura** e **consolidar/validar** os resultados. A leitura contém espera de I/O; a assinatura é deliberadamente implementada como cálculo puro em Python.

Nesta aula, a pergunta central é:

> **Uma única estratégia de paralelismo serve para leitura e cálculo?**

### Capacidades mobilizadas
- representar dependências por um grafo de tarefas (DAG);
- distinguir **Fork-Join** e **SPMD**;
- classificar etapas como **I/O-bound** ou **CPU-bound**;
- selecionar `ThreadPoolExecutor` ou `ProcessPoolExecutor` de forma justificada;
- implementar com **no máximo 2 workers**;
- validar equivalência com baseline sequencial;
- registrar hipótese, evidência, overhead e limitações do ambiente.

### Ancoragem da aula
Material alinhado ao **Plano de Ensino FO-178 2026/2 — Aula 2**, ao **PPC do CST em Inteligência Artificial — Anexo I**, à **Metodologia SENAI de Educação Profissional (MSEP)** e às referências da unidade curricular. O roteiro da aula orienta dados sintéticos, `max_workers=2`, funções de worker no nível de módulo e validação automática de equivalência.

### Versão do professor / solução comentada
Além das soluções, este notebook contém **interpretações esperadas e pontos de mediação**. Os tempos impressos não são gabarito; a correção é dada por **equivalência + justificativa coerente com o ambiente**.

## 0. Como usar este notebook — Google Colab e Visual Studio Code

### Google Colab
1. Faça upload do `.ipynb` no Colab ou abra-o pelo Google Drive.
2. Use **Ambiente de execução → Executar tudo** apenas depois de ler os enunciados marcados como **ATIVIDADE**.
3. A prática usa somente a **biblioteca-padrão do Python**; não há `pip install` obrigatório.
4. Para a parte com processos, o notebook cria um arquivo `.py`. No Colab, ele aparecerá no painel **Arquivos** à esquerda.
5. O script é executado pelo próprio interpretador do ambiente usando `subprocess`, o que torna o exemplo mais portátil do que definir workers diretamente em células interativas.

### Visual Studio Code
1. Instale/ative as extensões **Python** e **Jupyter**.
2. Abra o `.ipynb` e selecione um kernel Python 3.
3. Execute as células em ordem.
4. O arquivo `.py` criado na parte de processos aparecerá no **Explorer** da mesma pasta do notebook; você pode abri-lo, editar e salvar normalmente.
5. O notebook chamará esse script com o mesmo Python do kernel (`sys.executable`).

### Regra da aula
Usaremos **`MAX_WORKERS = 2`** nos exemplos dos estudantes. Os tempos são evidências do **seu ambiente e desta carga**, não uma regra universal.

In [ ]:
from __future__ import annotations

import importlib.util
import os
import platform
import statistics
import subprocess
import sys
import sysconfig
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

MAX_WORKERS = 2
DATA_DIR = Path.cwd() / "aula2_dados_sinteticos"

print("Diretório de trabalho:", Path.cwd())
print("MAX_WORKERS:", MAX_WORKERS)

In [ ]:
def diagnosticar_ambiente() -> dict:
    cpu_count_fn = getattr(os, "process_cpu_count", os.cpu_count)
    cpu_count = cpu_count_fn() or 1
    supports_ft = sysconfig.get_config_var("Py_GIL_DISABLED") == 1
    gil_check = getattr(sys, "_is_gil_enabled", None)
    gil_enabled = gil_check() if callable(gil_check) else None
    ambiente = "Google Colab" if importlib.util.find_spec("google.colab") else "Jupyter/VS Code ou outro"
    return {
        "ambiente": ambiente,
        "implementacao": platform.python_implementation(),
        "python": platform.python_version(),
        "cpus_logicas_visiveis": cpu_count,
        "build_free_threaded": supports_ft,
        "gil_ativo_detectavel": gil_enabled,
    }

info = diagnosticar_ambiente()
for chave, valor in info.items():
    print(f"{chave}: {valor}")

### Leitura didática do diagnóstico
- Em CPython tradicional com GIL ativo, threads tendem a ser úteis para **sobrepor espera de I/O**.
- Para laços puros em Python, processos podem explorar mais de um núcleo, mas pagam criação, serialização e comunicação.
- Builds free-threaded existem; por isso, a formulação correta é **condicionada ao ambiente**, e não uma regra absoluta.

**Mediação:** peça a hipótese antes de qualquer benchmark.

---
# Prática 1 — solução
## 1. Classificação e DAG

| Etapa | Classificação | Justificativa |
|---|---|---|
| Listar caminhos | coordenação/leve | custo pequeno no lote didático |
| Ler bytes | I/O-bound | pode ficar aguardando armazenamento/rede |
| Consultar metadados remotos | I/O-bound | espera de serviço/rede |
| Calcular assinatura | CPU-bound | laços Python e operações inteiras |
| Ordenar/consolidar | coordenação/leve | conjunto pequeno |
| Validar | coordenação/leve | comparação dos resultados |

### DAG textual possível

```text
listar caminhos
   ├── ler item 0 ──> assinatura 0 ──┐
   ├── ler item 1 ──> assinatura 1 ──┤
   ├── ...                           ├──> consolidar ──> validar
   └── ler item 7 ──> assinatura 7 ──┘
```

Os ramos de itens diferentes são candidatos à concorrência; dentro de cada item existe a dependência **leitura → cálculo**.

## 2. Matriz de decisão — solução possível

| Etapa | Workload | Modelo | Executor | Evidência | Risco/overhead |
|---|---|---|---|---|---|
| Leitura | I/O-bound | Fork-Join | `ThreadPoolExecutor` | tempo + `assert` | excesso de threads, latência artificial |
| Assinatura | CPU-bound pura em Python | SPMD dentro de região Fork-Join | `ProcessPoolExecutor` | tempo + `assert` | criação, pickle, transferência, granularidade |
| Consolidação | leve | Join | sequencial | ordem/correção | baixo |

Outras escolhas podem ser defensáveis se respeitarem dependências, correção e evidências.

In [ ]:
def criar_arquivos_sinteticos(diretorio: Path, quantidade: int = 8, tamanho: int = 4096) -> list[Path]:
    diretorio.mkdir(parents=True, exist_ok=True)
    for antigo in diretorio.glob("amostra_*.bin"):
        antigo.unlink()

    caminhos = []
    for indice in range(quantidade):
        dados = bytes((indice * 17 + offset * 31) % 256 for offset in range(tamanho))
        caminho = diretorio / f"amostra_{indice:02d}.bin"
        caminho.write_bytes(dados)
        caminhos.append(caminho)
    return caminhos

caminhos = criar_arquivos_sinteticos(DATA_DIR)
print("Arquivos criados:", len(caminhos))
print("Primeiros:", [p.name for p in caminhos[:3]])
print("Tamanho do primeiro arquivo:", caminhos[0].stat().st_size, "bytes")

In [ ]:
def medir(funcao, repeticoes: int = 3):
    # Executa a função algumas vezes e devolve (último_resultado, mediana_em_segundos).
    tempos = []
    resultado = None
    for _ in range(repeticoes):
        inicio = time.perf_counter()
        resultado = funcao()
        tempos.append(time.perf_counter() - inicio)
    return resultado, statistics.median(tempos)

In [ ]:
def read_sample(path: Path) -> tuple[str, bytes]:
    # Representa uma leitura com espera de armazenamento/rede.
    time.sleep(0.08)
    return path.name, path.read_bytes()


def cpu_signature(item: tuple[str, bytes], rounds: int = 1200) -> tuple[str, int]:
    # Carga CPU-bound pura em Python. Mesma entrada -> mesma saída.
    name, data = item
    acc = 2166136261
    for _ in range(rounds):
        for byte in data[:512]:
            acc ^= byte
            acc = (acc * 16777619) & 0xFFFFFFFF
    return name, acc

## 3. Baseline de I/O e versão com threads

In [ ]:
io_seq, t_io_seq = medir(lambda: [read_sample(caminho) for caminho in caminhos])


def executar_io_thread():
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        return list(executor.map(read_sample, caminhos))

io_thread, t_io_thread = medir(executar_io_thread)
assert io_seq == io_thread

print(f"I/O sequencial: {t_io_seq:.3f} s")
print(f"I/O threads (2 workers): {t_io_thread:.3f} s")
print("✅ Equivalência confirmada.")

### Interpretação esperada
A espera não desapareceu; parte dela pôde ser **sobreposta**. O objetivo didático não é calcular speedup formal, mas relacionar o comportamento ao tipo de workload e preservar a correção.

## 4. `map` versus `submit`/`as_completed`

In [ ]:
def read_sample_variable(index_and_path: tuple[int, Path]):
    indice, path = index_and_path
    atraso = 0.02 + (7 - indice) * 0.015
    time.sleep(atraso)
    return indice, path.name, len(path.read_bytes()), atraso

pares = list(enumerate(caminhos))

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futuros = {executor.submit(read_sample_variable, par): par[0] for par in pares}
    conclusao = [future.result() for future in as_completed(futuros)]

conclusao_ordenada = sorted(conclusao, key=lambda resultado: resultado[0])

print("Ordem de conclusão:", [r[0] for r in conclusao])
print("Ordem restaurada:   ", [r[0] for r in conclusao_ordenada])
assert [r[0] for r in conclusao_ordenada] == list(range(len(caminhos)))

### Ponto de mediação
`as_completed` é útil quando queremos reagir conforme cada tarefa termina, inclusive para tratamento individual de erros. Porém, **ordem de conclusão não é ordem de entrada**. Se a semântica exige ordem, a identidade original precisa acompanhar o resultado ou ser reconstruída.

---
# Pós-intervalo — CPU-bound, GIL e processos
## 5. Baseline CPU e threads

In [ ]:
cpu_seq, t_cpu_seq = medir(lambda: [cpu_signature(item) for item in io_seq])


def executar_cpu_thread():
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        return list(executor.map(cpu_signature, io_seq))

cpu_thread, t_cpu_thread = medir(executar_cpu_thread)
assert cpu_seq == cpu_thread

print(f"CPU sequencial: {t_cpu_seq:.3f} s")
print(f"CPU threads:    {t_cpu_thread:.3f} s")
print("✅ Resultados equivalentes.")

### Interpretação esperada
Em uma build CPython tradicional com GIL ativo, threads não costumam escalar laços de bytecode Python em múltiplos núcleos. Isso é **hipótese condicionada ao ambiente**; extensões podem liberar o GIL e builds free-threaded mudam a premissa.

## 6. Script portátil com `ProcessPoolExecutor`

Criamos um arquivo `.py` porque o padrão portátil exige workers importáveis/no nível do módulo e proteção do ponto de entrada com `if __name__ == "__main__"`.

O script mede:
- pipeline completo sequencial;
- I/O sequencial × threads;
- CPU sequencial × threads × processos;
- pipeline threads(I/O) + processos(CPU);
- equivalência de todas as saídas.

In [ ]:
SCRIPT_SOLUCAO = r'''from __future__ import annotations

import os
import platform
import statistics
import sys
import sysconfig
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from pathlib import Path

MAX_WORKERS = 2
DATA_DIR = Path.cwd() / "aula2_dados_sinteticos"


def show_environment() -> None:
    cpu_count_fn = getattr(os, "process_cpu_count", os.cpu_count)
    cpu_count = cpu_count_fn() or 1
    supports_ft = sysconfig.get_config_var("Py_GIL_DISABLED") == 1
    gil_check = getattr(sys, "_is_gil_enabled", None)
    gil_enabled = gil_check() if callable(gil_check) else None
    print(platform.python_implementation(), platform.python_version())
    print("CPUs lógicas visíveis:", cpu_count)
    print("Build free-threaded:", supports_ft, "| GIL detectável/ativo:", gil_enabled)


def read_sample(path: Path) -> tuple[str, bytes]:
    time.sleep(0.08)
    return path.name, path.read_bytes()


def cpu_signature(item: tuple[str, bytes], rounds: int = 1200) -> tuple[str, int]:
    name, data = item
    acc = 2166136261
    for _ in range(rounds):
        for byte in data[:512]:
            acc ^= byte
            acc = (acc * 16777619) & 0xFFFFFFFF
    return name, acc


def medir(funcao, repeticoes: int = 3):
    tempos = []
    resultado = None
    for _ in range(repeticoes):
        inicio = time.perf_counter()
        resultado = funcao()
        tempos.append(time.perf_counter() - inicio)
    return resultado, statistics.median(tempos)


def io_sequencial(caminhos):
    return [read_sample(caminho) for caminho in caminhos]


def io_threads(caminhos):
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        return list(executor.map(read_sample, caminhos))


def cpu_sequencial(itens):
    return [cpu_signature(item) for item in itens]


def cpu_threads(itens):
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        return list(executor.map(cpu_signature, itens))


def cpu_processos(itens):
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        return list(executor.map(cpu_signature, itens))


def pipeline_sequencial(caminhos):
    return [cpu_signature(read_sample(caminho)) for caminho in caminhos]


def pipeline_paralelo(caminhos):
    lidos = io_threads(caminhos)
    return cpu_processos(lidos)


def main() -> None:
    show_environment()
    caminhos = sorted(DATA_DIR.glob("amostra_*.bin"))
    if not caminhos:
        raise RuntimeError("Execute primeiro a célula que cria os dados sintéticos.")

    base_pipeline, t_pipeline_seq = medir(lambda: pipeline_sequencial(caminhos))

    lidos_seq, t_io_seq = medir(lambda: io_sequencial(caminhos))
    lidos_thr, t_io_thr = medir(lambda: io_threads(caminhos))
    assert lidos_seq == lidos_thr

    cpu_seq, t_cpu_seq = medir(lambda: cpu_sequencial(lidos_seq))
    cpu_thr, t_cpu_thr = medir(lambda: cpu_threads(lidos_seq))
    cpu_proc, t_cpu_proc = medir(lambda: cpu_processos(lidos_seq))
    assert cpu_seq == cpu_thr == cpu_proc

    pipeline_proc, t_pipeline_proc = medir(lambda: pipeline_paralelo(caminhos))
    assert base_pipeline == pipeline_proc

    print("\nMedianas didáticas (segundos)")
    print(f"I/O sequencial:      {t_io_seq:.3f}")
    print(f"I/O threads:         {t_io_thr:.3f}")
    print(f"CPU sequencial:      {t_cpu_seq:.3f}")
    print(f"CPU threads:         {t_cpu_thr:.3f}")
    print(f"CPU processos:       {t_cpu_proc:.3f}")
    print(f"Pipeline sequencial: {t_pipeline_seq:.3f}")
    print(f"Pipeline 2 estágios: {t_pipeline_proc:.3f}")
    print("Validação: todas as comparações de equivalência passaram.")


if __name__ == "__main__":
    main()
'''

script_path = Path.cwd() / "aula2_processos_solucao.py"
script_path.write_text(SCRIPT_SOLUCAO, encoding="utf-8")
print("Script criado:", script_path)

In [ ]:
resultado = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=Path.cwd(),
    text=True,
    capture_output=True,
    check=False,
)

print(resultado.stdout)
if resultado.returncode != 0:
    print(resultado.stderr)
    raise RuntimeError(f"O script terminou com código {resultado.returncode}")

### Interpretação do professor
O esperado no experimento didático é:

- **I/O:** threads frequentemente reduzem o tempo por sobreposição de espera;
- **CPU pura em Python:** com GIL tradicional ativo, threads tendem a não mostrar ganho consistente;
- **processos:** podem usar múltiplos núcleos, mas o ganho depende da granularidade e dos custos de criação/serialização;
- **pipeline:** o resultado só é aceito porque o `assert` demonstra equivalência com o baseline.

Se processos perderem para o sequencial, não trate como “erro do aluno”: peça uma hipótese sobre **overhead versus quantidade de trabalho útil**.

## 7. SPMD — blocos contíguos e distribuição cíclica

In [ ]:
nomes = [p.name for p in caminhos]

def particionar_blocos(itens: list[str], workers: int) -> list[list[str]]:
    tamanho = len(itens) // workers
    particoes = []
    for worker in range(workers):
        inicio = worker * tamanho
        fim = len(itens) if worker == workers - 1 else (worker + 1) * tamanho
        particoes.append(itens[inicio:fim])
    return particoes


def particionar_ciclico(itens: list[str], workers: int) -> list[list[str]]:
    return [itens[worker::workers] for worker in range(workers)]

blocos = particionar_blocos(nomes, MAX_WORKERS)
ciclicas = particionar_ciclico(nomes, MAX_WORKERS)

print("Blocos contíguos:")
for w, p in enumerate(blocos):
    print(f"  Worker {w}:", p)

print("\nDistribuição cíclica:")
for w, p in enumerate(ciclicas):
    print(f"  Worker {w}:", p)

# Invariantes: cobertura completa e sem duplicação
for particoes in (blocos, ciclicas):
    achatada = [item for p in particoes for item in p]
    assert sorted(achatada) == sorted(nomes)
    assert len(achatada) == len(set(achatada))

print("\n✅ As duas estratégias cobrem todos os itens sem sobreposição.")

### Discussão
Blocos contíguos são simples de reconstruir. A distribuição cíclica pode ajudar quando custos se alternam entre itens, mas a escolha depende do padrão de carga. Aqui o objetivo é identificar o princípio SPMD: **mesmo programa, dados/partições distintas, resultados identificados e reunidos ao final**.

## 8. Exemplo curto de `Future` e tratamento individual

Este exemplo não altera o pipeline principal; ele serve para tornar visível a abstração `Future`.

In [ ]:
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(read_sample, caminho) for caminho in caminhos[:4]]
    print("Quantidade de Future criados:", len(futures))
    print("Estado imediatamente após submit:", [(f.done(), f.running()) for f in futures])
    resultados = [f.result() for f in futures]

print("Resultados recebidos:", [nome for nome, _ in resultados])

## 9. Síntese — resposta esperada da justificativa técnica

Uma boa resposta do estudante deve dizer, em essência:

- O **DAG** revela ramos independentes por item e dependência leitura → cálculo no mesmo item.
- O fluxo do lote pode ser organizado como **Fork-Join**; a etapa de cálculo também pode ser descrita como **SPMD** quando workers executam o mesmo código sobre partições diferentes.
- `ThreadPoolExecutor` é uma hipótese coerente para I/O porque a espera pode ser sobreposta.
- `ProcessPoolExecutor` é uma hipótese coerente para CPU-bound pura em Python porque processos possuem interpretadores/GILs próprios, mas há custo de serialização, criação e comunicação.
- `map` preserva a correspondência posicional das entradas; `as_completed` expõe ordem de conclusão e pode exigir reconstrução da ordem.
- Nenhum ganho de desempenho é aceito sem validar a **equivalência** com o baseline e sem limitar a conclusão ao ambiente medido.

## Checklist formativo do docente

- [x] Hipótese registrada antes do teste.
- [x] DAG coerente com dependências.
- [x] Workload I/O/CPU classificado.
- [x] Modelo e executor justificados.
- [x] `MAX_WORKERS = 2`.
- [x] `ThreadPoolExecutor` testado com equivalência.
- [x] `submit`/`Future`/`as_completed` observados.
- [x] Ordem tratada explicitamente.
- [x] `ProcessPoolExecutor` executado por `.py` portátil.
- [x] Worker top-level + `if __name__ == "__main__"`.
- [x] SPMD demonstrado por partições.
- [x] Overhead e limites do ambiente registrados.

## Referências da aula

- **SENAI-GO.** Plano de Ensino FO-178 — Computação Paralela, 2026/2. Aula 2: Fork-Join, SPMD, grafos de tarefas, threads, processos, executores, CPU-bound/I/O-bound e GIL.
- **SENAI-GO.** Projeto Pedagógico do Curso Superior de Tecnologia em Inteligência Artificial, 2026. Anexo I — Unidade Curricular Computação Paralela.
- **SENAI/DN.** *Metodologia SENAI de Educação Profissional*. Brasília: SENAI/DN, 2019.
- **BORDIN, Maycon V. et al.** *Processamento Paralelo e Distribuído*. Grupo A, 2021.
- **SILVA, G. P.; BIANCHINI, C. P.; COSTA, E. B.** *Programação paralela e distribuída: com MPI, OpenMP e OpenACC para computação de alto desempenho*. Casa do Código, 2022.
- **TANENBAUM, Andrew S.** *Sistemas Operacionais Modernos*. Pearson, 2009.
- **Python Software Foundation.** Documentação oficial de `concurrent.futures`, processos/threads e GIL, conforme documentação técnica indicada no Plano de Ensino e no roteiro da aula.

> Observação: este notebook não aprofunda locks, semáforos, race conditions, deadlocks, NUMA, MPI, GPU, leis formais de speedup ou profiling avançado, pois esses conteúdos pertencem a outros encontros do plano.